In [21]:
import pandas as pd
import requests
import re
import time
import os
from duckduckgo_search import DDGS
from tqdm import tqdm

In [ ]:
# --- CONFIGURATION ---
INPUT_CSV = "Top-1000-Brasil.csv"
OUTPUT_CSV = "companies_domains_validated_2.csv"
SLEEP_BETWEEN_QUERIES = 2  # in seconds
MAX_RETRIES = 3
AUTO_SAVE_EVERY = 10  # companies

In [34]:
# --- LOAD COMPANY LIST ---
df_companies = pd.read_csv(INPUT_CSV)
company_names = df_companies['Company'].tolist()

In [49]:
df_companies

,rank_2023,Company,headquarters,sector,net-revenue-in-r$-millions,net-income-in-r$-millions,net-margin-%,return-on-average-equity-%
0,1,Petrobras,RJ,Petróleo e Gás,"511.994,00","125.166,00","24,4","33,5"
1,2,JBS,SP,Alimentos e Bebidas,"363.816,50","-729,1","-0,2","-1,5"
2,3,Raízen,RJ,Petróleo e Gás,"220.454,20","614,1","0,3","2,7"
3,4,Vale,RJ,Mineração,"208.066,00","40.554,00","19,5","20,6"
4,5,Vibra,RJ,Petróleo e Gás,"162.947,00","4.766,00","2,9","33,6"
...,...,...,...,...,...,...,...,...
995,996,Santa Casa de Misericórdia de Belo Horizonte,MG,Serviços Médicos,"738,8","58,4","7,9","25,9"
996,997,Grupo Nortis,SP,Empreendimentos Imobiliários,"737,9","73,5",10,"38,1"
997,998,Leitesol,SP,Alimentos e Bebidas,"737,5","56,3","7,6","36,8"
998,999,SAAM,RJ,Transportes e Logística,"735,5","151,2","20,6","22,8"


In [36]:
# --- SEARCH DOMAIN USING DUCKDUCKGO ---
def search_domain(company_name):
    query = f"{company_name} site oficial da empresa no brasil"
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query)
            for result in results:
                url = result.get("href")
                if url:
                    domain = re.findall(r"https?://(www\.)?([^/]+)", url)
                    if domain:
                        return domain[0][1]
    except Exception as e:
        print(f"[ERROR] Search failed for '{company_name}': {e}")
    return None


# --- VALIDATE DOMAIN EXISTS (HTTPS) ---
def validate_domain(domain):
    try:
        url = f"https://{domain}"
        response = requests.get(url, timeout=5)
        return response.status_code < 400
    except Exception:
        return False

# --- PROCESS SINGLE COMPANY WITH RETRIES ---
def process_company(company_name):
    for attempt in range(1, MAX_RETRIES + 1):
        domain = search_domain(company_name)
        if domain:
            is_valid = validate_domain(domain)
            return {
                "Company": company_name,
                "Domain": domain,
                "Domain_Valid": is_valid
            }
        else:
            print(f"[RETRY] Attempt {attempt} failed for '{company_name}'. Retrying...")
            time.sleep(SLEEP_BETWEEN_QUERIES)
    print(f"[FAILED] Could not resolve domain for '{company_name}' after {MAX_RETRIES} attempts.")
    return {
        "Company": company_name,
        "Domain": "",
        "Domain_Valid": False
    }


In [37]:

# --- IF PARTIAL OUTPUT EXISTS, CONTINUE FROM LAST POINT ---
processed = []
start_index = 0
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    processed_companies = set(existing['Company'])
    start_index = len(existing)
    processed = existing.to_dict(orient='records')
    print(f"[INFO] Resuming from company #{start_index}...")

# --- MAIN LOOP ---
for idx in tqdm(range(start_index, len(company_names)), desc="Processing companies"):
    company_name = company_names[idx]
    result = process_company(company_name)
    processed.append(result)
    
    # Save after every N companies
    if (idx + 1) % AUTO_SAVE_EVERY == 0 or idx == len(company_names) - 1:
        pd.DataFrame(processed).to_csv(OUTPUT_CSV, index=False)
        print(f"[AUTO-SAVE] Saved progress at {idx + 1} companies.")
    
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\n✅ Process completed. Final results saved to: {OUTPUT_CSV}")


Processing companies:   0%|          | 0/1000 [00:00<?, ?it/s]

[ERROR] Search failed for 'Petrobras': https://lite.duckduckgo.com/lite/ 202 Ratelimit
[RETRY] Attempt 1 failed for 'Petrobras'. Retrying...
[ERROR] Search failed for 'Petrobras': https://lite.duckduckgo.com/lite/ 202 Ratelimit
[RETRY] Attempt 2 failed for 'Petrobras'. Retrying...


Processing companies:   0%|          | 0/1000 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [51]:
df_empresas = pd.read_csv('Top-100-Brasil.csv')
df_dominios = pd.read_csv('companies_domains_validated.csv')

# Junção dos dados com base no nome da empresa (Company)
df_final = pd.merge(df_empresas, df_dominios, on='Company', how='left')

# (Opcional) Exibir empresas que não tiveram domínio associado
na_domains = df_final[df_final['Domain'].isna()]
if not na_domains.empty:
    print("Warning: May have companies with ou assigned domain")
    print(na_domains[['Company']])

# Salvar o resultado final em um novo CSV
df_final.to_csv('empresas_com_dominios.csv', index=False)

In [52]:
# Dicionário de mapeamento de setores originais para categorias agregadas
setor_agregado = {
    "Comércio Varejista": "Comércio",
    "Comércio Atacadista e Exterior": "Comércio",
    "Serviços Financeiros": "Serviços Financeiros e Corporativos",
    "Serviços Especializados": "Serviços Financeiros e Corporativos",
    "Transportes e Logística": "Transporte e Logística",
    "TI & Telecom": "Tecnologia e Comunicação",
    "Comunicação e Gráfica": "Tecnologia e Comunicação",
    "Eletroeletrônica": "Tecnologia e Comunicação",
    "Serviços Médicos": "Saúde e Bem-Estar",
    "Farmacêutica e Cosméticos": "Saúde e Bem-Estar",
    "Educação": "Educação",
    "Petróleo e Gás": "Energia",
    "Energia Elétrica": "Energia",
    "Bioenergia": "Energia",
    "Construção e Engenharia": "Construção e Infraestrutura",
    "Mat. de Constr. e de Acabamento": "Construção e Infraestrutura",
    "Empreendimentos Imobiliários": "Construção e Infraestrutura",
    "Plásticos e Borracha": "Indústria de Transformação",
    "Química e Petroquímica": "Indústria de Transformação",
    "Papel e Celulose": "Indústria de Transformação",
    "Metalurgia e Siderurgia": "Indústria Pesada",
    "Mecânica": "Indústria Pesada",
    "Mineração": "Indústria Pesada",
    "Alimentos e Bebidas": "Agroindústria e Alimentos",
    "Agronegócio": "Agroindústria e Alimentos",
    "Fumo": "Agroindústria e Alimentos",
    "Indústria da Moda": "Bens de Consumo e Duráveis",
    "Veículos e Peças": "Bens de Consumo e Duráveis",
    "Água, Saneamento e Serviços Ambientais": "Serviços Públicos e Ambientais"
}

# Leitura do arquivo original
df = pd.read_csv("empresas_com_dominios.csv")

# Aplicar o de-para usando a coluna 'sector'
df["aggregated_sector"] = df["sector"].map(setor_agregado)

# Avisar se houver setores não mapeados
na_setores = df[df["aggregated_sector"].isna()]
if not na_setores.empty:
    print("Atenção: alguns setores não foram mapeados:")
    print(na_setores["sector"].unique())

# Salvar o novo DataFrame em um arquivo CSV
df.to_csv("empresas_com_setor_agregado.csv", index=False)

In [55]:
# Dicionário de mapeamento: setor original → macrosetor
macrosetor_map = {
    # Comércio e Serviços
    "Comércio Varejista": "Comércio e Serviços",
    "Comércio Atacadista e Exterior": "Comércio e Serviços",
    "Serviços Financeiros": "Comércio e Serviços",
    "Serviços Especializados": "Comércio e Serviços",
    "Serviços Médicos": "Comércio e Serviços",
    "Farmacêutica e Cosméticos": "Comércio e Serviços",
    "Educação": "Comércio e Serviços",

    # Indústria
    "Indústria da Moda": "Indústria",
    "Veículos e Peças": "Indústria",
    "Metalurgia e Siderurgia": "Indústria",
    "Mecânica": "Indústria",
    "Mineração": "Indústria",
    "Plásticos e Borracha": "Indústria",
    "Química e Petroquímica": "Indústria",
    "Papel e Celulose": "Indústria",

    # Infraestrutura e Logística
    "Construção e Engenharia": "Infraestrutura e Logística",
    "Mat. de Constr. e de Acabamento": "Infraestrutura e Logística",
    "Empreendimentos Imobiliários": "Infraestrutura e Logística",
    "Transportes e Logística": "Infraestrutura e Logística",
    "Energia Elétrica": "Infraestrutura e Logística",
    "Bioenergia": "Infraestrutura e Logística",
    "Petróleo e Gás": "Infraestrutura e Logística",
    "Água, Saneamento e Serviços Ambientais": "Infraestrutura e Logística",

    # Tecnologia e Comunicação
    "TI & Telecom": "Tecnologia e Comunicação",
    "Comunicação e Gráfica": "Tecnologia e Comunicação",
    "Eletroeletrônica": "Tecnologia e Comunicação",

    # Agropecuária e Alimentos
    "Agronegócio": "Agropecuária e Alimentos",
    "Fumo": "Agropecuária e Alimentos",
    "Alimentos e Bebidas": "Agropecuária e Alimentos"
}

# Leitura do arquivo CSV original
df = pd.read_csv("empresas_com_setor_agregado.csv")

# Aplicação do mapeamento para gerar nova coluna
df["macro_sector"] = df["sector"].map(macrosetor_map)

# Avisar se há setores sem mapeamento
na_macro = df[df["macro_sector"].isna()]
if not na_macro.empty:
    print("Setores não mapeados encontrados:")
    print(na_macro["sector"].unique())

# Salvar em novo CSV
df.to_csv("empresas_com_macrosetor.csv", index=False)

# União Europeia

In [18]:
# --- CONFIGURATION ---
INPUT_CSV = "Top-1000-EU.csv"
OUTPUT_CSV = "eu_companies_domains_validated.csv"
SLEEP_BETWEEN_QUERIES = 1  # in seconds
MAX_RETRIES = 3
AUTO_SAVE_EVERY = 10  # companies

In [19]:
# --- LOAD COMPANY LIST ---
df_companies = pd.read_csv(INPUT_CSV)
company_names = df_companies['company'].tolist()

In [20]:
def search_domain(company_name):
    query = f"{company_name} official homeland website"
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query)
            for result in results:
                url = result.get("href")
                if url:
                    domain = re.findall(r"https?://(www\.)?([^/]+)", url)
                    if domain:
                        return domain[0][1]
    except Exception as e:
        print(f"[ERROR] Search failed for '{company_name}': {e}")
    return None

# --- IF PARTIAL OUTPUT EXISTS, CONTINUE FROM LAST POINT ---
processed = []
start_index = 0
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    processed_companies = set(existing['Company'])
    start_index = len(existing)
    processed = existing.to_dict(orient='records')
    print(f"[INFO] Resuming from company #{start_index}...")

# --- MAIN LOOP ---
for idx in tqdm(range(start_index, len(company_names)), desc="Processing companies"):
    company_name = company_names[idx]
    result = process_company(company_name)
    processed.append(result)
    
    # Save after every N companies
    if (idx + 1) % AUTO_SAVE_EVERY == 0 or idx == len(company_names) - 1:
        pd.DataFrame(processed).to_csv(OUTPUT_CSV, index=False)
        print(f"[AUTO-SAVE] Saved progress at {idx + 1} companies.")
    
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\n✅ Process completed. Final results saved to: {OUTPUT_CSV}")


Processing companies:   0%|          | 0/1000 [00:00<?, ?it/s]

[ERROR] Search failed for 'SAP': https://lite.duckduckgo.com/lite/ 202 Ratelimit
[RETRY] Attempt 1 failed for 'SAP'. Retrying...
[ERROR] Search failed for 'SAP': https://lite.duckduckgo.com/lite/ 202 Ratelimit
[RETRY] Attempt 2 failed for 'SAP'. Retrying...


Processing companies:   0%|          | 0/1000 [00:05<?, ?it/s]


KeyboardInterrupt: 

## Incidentes Relacionados ao DNS (2000–2025)

| Ano  | Incidente                                  | Tipo de Incidente               | Setor Afetado                    | Impacto                 | Entidade Afetada                          | Região                 |
|------|--------------------------------------------|----------------------------------|----------------------------------|-------------------------|--------------------------------------------|------------------------|
| 2002 | Ataque DDoS aos Root Servers               | DDoS                             | Infraestrutura                   | Estratégico             | Servidores raiz do DNS                    | Global                 |
| 2005 | Pharming em bancos brasileiros             | Cache poisoning                  | Financeiro                       | Econômico / Estratégico | HSBC, Itaú, etc.                          | Brasil                 |
| 2005 | Sequestro do domínio Panix.com             | Domain hijacking                 | Telecomunicações                 | Econômico / Estratégico | Panix (ISP)                               | EUA                    |
| 2006 | Ataque DDoS à Blue Security                | DDoS                             | Segurança / Mídia                | Econômico / Estratégico | Blue Security / Six Apart                 | Israel / EUA           |
| 2007 | DDoS aos Root Servers                      | DDoS                             | Infraestrutura                   | Estratégico             | Instâncias G, L, M dos servidores raiz    | Global                 |
| 2007 | Malware DNSChanger (Ghost Click)           | DNS hijacking via malware        | Usuários / Governo               | Econômico / Estratégico | Rove Digital                              | Global                 |
| 2008 | Hijack Photobucket                         | DNS hijacking                    | Mídia Social                     | Estratégico             | Photobucket                               | Global                 |
| 2008 | Hijack ICANN/IANA                          | DNS hijacking                    | Governança Internet              | Estratégico             | ICANN / IANA                              | Global                 |
| 2008 | Hijack CheckFree                           | DNS hijacking                    | Financeiro                       | Econômico / Estratégico | CheckFree (Fiserv)                        | EUA                    |
| 2008 | Bug de Kaminsky                            | Cache poisoning                  | Infraestrutura DNS               | Estratégico             | Servidores DNS globais                    | Global                 |
| 2008 | Hijack Comcast.net                         | Domain hijacking                 | Telecomunicações                 | Econômico / Estratégico | Comcast                                   | EUA                    |
| 2008 | Apagão do Speedy (Telefônica)              | Falha técnica (pane DNS)         | Telecomunicações (ISP)           | Econômico / Estratégico | Telefônica (Speedy)                       | São Paulo, Brasil      |
| 2009 | Hijack de domínios .nz (Domainz)           | DNS hijacking                    | Corporativo (.co.nz)             | Estratégico             | Microsoft, Coca-Cola, etc.                | Nova Zelândia          |
| 2009 | Pharming Bradesco via NET Virtua           | Cache poisoning                  | Financeiro                       | Econômico / Estratégico | Bradesco, NET Virtua                      | Brasil                 |
| 2009 | Hijack do Twitter.com                      | DNS hijacking                    | Mídia Social                     | Estratégico             | Twitter                                   | Global / Irã           |
| 2010 | Hijack do Baidu.com                        | DNS hijacking                    | Tecnologia                       | Estratégico             | Baidu                                     | China                  |
| 2011 | Ataque generalizado (pharming DNS)         | Cache poisoning / malware        | Multissetorial (consumidores)    | Estratégico             | Vários provedores e roteadores domésticos | Brasil                 |
| 2012 | Queda do Google.com.br                     | Erro de configuração DNS         | Serviços de Internet             | Estratégico             | Google Brasil                             | Brasil                 |
| 2012 | Comprometimento de 4.5M modems DSL         | Exploração de roteadores         | Telecom / Bancos                 | Econômico / Estratégico | Clientes de ISPs; bancos visados          | Brasil                 |
| 2013 | Ataque ao Spamhaus                         | DNS amplification                | Infraestrutura                   | Econômico / Estratégico | Spamhaus / CloudFlare                     | Global                 |
| 2014 | Hijack de DNS na Net Virtua (Flash falso)  | DNS hijacking                    | Telecom / Usuários               | Estratégico             | Net Virtua                                | Brasil                 |
| 2015 | DDoS aos Root Servers                      | DDoS                             | Infraestrutura                   | Estratégico             | Servidores raiz do DNS                    | Global                 |
| 2016 | Ataque ao Dyn                              | DDoS (IoT/Mirai)                 | Infraestrutura / Web             | Econômico / Estratégico | Dyn / Amazon / Twitter / Spotify          | EUA                    |
| 2016 | Sequestro total DNS de banco (36 domínios) | Domain hijacking                 | Financeiro                       | Econômico / Estratégico | Banco (prov. Banrisul)                    | Brasil                 |
| 2018 | Campanha DNSpionage                        | DNS hijacking                    | Governo / Energia                | Estratégico             | Entidades no Oriente Médio                | Oriente Médio          |
| 2018 | Botnet GhostDNS (100k+ roteadores)         | DNS changer botnet               | Telecom / Bancos                 | Econômico / Estratégico | Usuários domésticos; bancos               | Brasil                 |
| 2019 | Campanha Sea Turtle                        | DNS hijacking                    | Governo / TLDs                   | Estratégico             | Vários TLDs / Infraestrutura              | Oriente Médio / Global |
| 2020 | Hijack de exchanges cripto (GoDaddy)       | DNS hijacking                    | Financeiro / Cripto              | Econômico               | Liquid / NiceHash / GoDaddy               | Global                 |
| 2020 | Falha de DNSSEC no .br                     | Erro de operação DNSSEC          | Infraestrutura de TLD            | Estratégico             | NIC.br / Registro.br                      | Brasil                 |
| 2021 | Ataque à VoIP.ms                           | DDoS                             | Telecomunicações                 | Econômico               | VoIP.ms                                   | América Latina / EUA   |
| 2021 | Hijack PancakeSwap / Cream Finance         | DNS hijacking                    | Finanças descentralizadas (DeFi) | Econômico               | PancakeSwap / Cream Finance               | Global                 |
| 2021 | Ataque a órgãos federais via DNS           | Falha técnica / vetor de ataque  | Governo Federal                  | Estratégico             | Ministério da Saúde e outros              | Brasil                 |
| 2024 | Estudo 'Sitting Ducks' (delegações fracas) | Configuração fraca               | Diversos setores                 | Econômico / Estratégico | 35 mil domínios analisados                | Global                 |


# Incidentes Relacionados a DNS no Brasil (2000–2025)

A seguir é apresentada uma listagem detalhada de incidentes envolvendo DNS (Domain Name System) no Brasil entre 2000 e 2025. Incluem-se ataques intencionais (DDoS, hijacking, spoofing etc.), falhas operacionais, erros de configuração, interrupções de serviço, más práticas e outros eventos relevantes – abrangendo tanto grandes incidentes quanto casos menores. Cada incidente traz um breve resumo e informações estruturadas: tipo, setor afetado, impacto, entidade envolvida, região, ano e fontes (incluindo materiais técnicos/acadêmicos quando disponíveis).

---

## 2008: Apagão do Speedy – Falha de DNS da Telefônica (São Paulo)

**Resumo:**  
Em julho de 2008, o serviço de banda larga Speedy (Telefônica) sofreu um apagão generalizado no Estado de São Paulo. Milhões de usuários ficaram sem internet por horas. Investigações apontaram problemas nos servidores DNS da Telefônica, evidenciando a dependência crítica do DNS no acesso à Internet no Brasil.

- **Tipo de incidente:** Falha técnica/operacional (pane nos servidores DNS).  
- **Setor afetado:** Telecomunicações (ISP) – impacto cascata em serviços públicos e privados.  
- **Impacto:** Econômico (queda de serviço em massa, prejuízos e ressarcimentos) e estratégico (alerta sobre infraestrutura crítica).  
- **Entidade afetada:** Telefônica (serviço Speedy, atualmente Vivo Fibra).  
- **Região:** Estado de São Paulo (especialmente capital e região metropolitana).  
- **Ano:** 2008.  
- **Fonte:** [Infowester](https://www.infowester.com), [SuperInteressante](https://super.abril.com.br).  

---

## 2011: Ataque Generalizado – Envenenamento de DNS e Pharming em Massa

**Resumo:**  
No final de 2011, o Brasil foi alvo de um dos maiores ataques de pharming já registrados no país, com sequestro de DNS em escala nacional. Hackers comprometeram servidores DNS de provedores e roteadores domésticos, redirecionando usuários a sites falsos.

- **Tipo de incidente:** Ataque cibernético (envenenamento de cache DNS; sequestro de DNS via malware em roteadores).  
- **Setor afetado:** Usuários de múltiplos setores (consumidores em geral).  
- **Impacto:** Estratégico (distribuição massiva de malware, comprometimento da confiança em serviços online) e econômico (porta de entrada para fraudes financeiras).  
- **Entidades afetadas:** Vários provedores de internet brasileiros; milhões de roteadores DSL domésticos alterados.  
- **Região:** Nacional.  
- **Ano:** 2011.  
- **Fontes:** [SecurityWeek](https://www.securityweek.com), [Kaspersky Securelist](https://securelist.com).  

---

## 2012: Comprometimento em Massa de Modems DSL (4,5 milhões de roteadores)

**Resumo:**  
Entre 2011 e 2012, uma campanha contínua de ataques atingiu modems ADSL vulneráveis no Brasil, comprometendo mais de 4,5 milhões de roteadores domésticos. O tráfego DNS dos usuários infectados era redirecionado por servidores maliciosos, permitindo ataques de pharming.

- **Tipo de incidente:** Ataque cibernético (exploração de vulnerabilidade em roteadores; DNS hijacking automatizado).  
- **Setor afetado:** Telecomunicações / usuários domésticos, com foco em clientes de bancos.  
- **Impacto:** Econômico (fraudes bancárias, roubo de dados) e estratégico (exposição de fragilidades em infraestrutura de última milha).  
- **Entidades afetadas:** Vários modelos de modems DSL distribuídos por ISPs brasileiros.  
- **Região:** Nacional.  
- **Ano:** 2011–2012.  
- **Fontes:** [Securelist](https://securelist.com), [Radware](https://www.radware.com).  

---

## 2012: Queda do Google.com.br por Erro de DNS (Brasil)

**Resumo:**  
Em 26 de novembro de 2012, usuários em todo o Brasil relataram a queda repentina do Google e serviços relacionados. O problema foi rastreado a uma falha de configuração DNS, afetando o domínio google.com.br.

- **Tipo de incidente:** Erro técnico de DNS (falha de configuração ou propagação do zone file do Google).  
- **Setor afetado:** Serviços de Internet (busca, e-mail, vídeo – Big Tech).  
- **Impacto:** Estratégico, porém de curta duração – interrupção de serviços amplamente utilizados.  
- **Entidade afetada:** Google Brasil (domínio google.com.br e subserviços).  
- **Região:** Brasil.  
- **Ano:** 2012.  
- **Fontes:** [O Globo](https://oglobo.globo.com).  

---

## 2014: Sequestro de DNS na Net Virtua – “Flash Player Falso”

**Resumo:**  
Em abril de 2014, assinantes da Net Virtua enfrentaram um ataque de DNS hijacking. O DNS recursivo da Net foi alterado maliciosamente, redirecionando usuários para uma página que oferecia um falso instalador do Adobe Flash Player.

- **Tipo de incidente:** Ataque (sequestro de DNS de provedor, provavelmente via invasão ou envenenamento de cache).  
- **Setor afetado:** Telecomunicações (ISP) – usuários residenciais do Virtua; setor de tecnologia.  
- **Impacto:** Estratégico – distribuição de malware em massa, quebra de confiança em resolução DNS do ISP.  
- **Entidade afetada:** Net Virtua (atualmente Claro).  
- **Região:** São Paulo e possivelmente outros locais atendidos pela Net Virtua.  
- **Ano:** 2014.  
- **Fonte:** [Tecnoblog](https://tecnoblog.net).  

---

## 2016: Sequestro Total de DNS de um Banco (36 Domínios .br)

**Resumo:**  
Em 22 de outubro de 2016, hackers comprometeram o registro DNS de um grande banco brasileiro, alterando as entradas DNS de todos os seus 36 domínios. Clientes foram redirecionados para páginas fraudulentas, onde inseriram credenciais bancárias.

- **Tipo de incidente:** Ataque cibernético (sequestro de DNS via comprometimento de registrador).  
- **Setor afetado:** Financeiro (banco comercial de grande porte).  
- **Impacto:** Econômico (roubo massivo de credenciais bancárias) e estratégico (violação de confiança em serviços bancários online).  
- **Entidade afetada:** Banco brasileiro de varejo (possivelmente Banrisul).  
- **Região:** Brasil.  
- **Ano:** 2016.  
- **Fontes:** [Wired](https://www.wired.com), [Vice Brasil](https://www.vice.com).  

---

## 2018: Botnet GhostDNS – 100 mil Roteadores Hijacked (Ataque a Bancos)

**Resumo:**  
No segundo semestre de 2018, a botnet GhostDNS comprometeu mais de 100 mil roteadores residenciais no Brasil, redirecionando usuários para páginas falsas de bancos como Banco do Brasil e Itaú.

- **Tipo de incidente:** Ataque cibernético (botnet de DNS changer targeting IoT/roteadores).  
- **Setor afetado:** Usuários domésticos e setor bancário.  
- **Impacto:** Econômico (roubo de dados bancários) e estratégico (escala ampla, dificultando mitigação).  
- **Entidades afetadas:** Diversos modelos de roteadores domésticos vulneráveis; principais bancos brasileiros.  
- **Região:** Primariamente Brasil, com expansão global.  
- **Ano:** 2018.  
- **Fontes:** [The Register](https://www.theregister.com), [Radware](https://www.radware.com).  

---

## 2020: Incidente DNSSEC no .br – Chave Indevida Causa Falha de Resolução

**Resumo:**  
Em 3 de agosto de 2020, diversos sites .br ficaram inacessíveis devido a um erro de assinatura DNSSEC. Registros foram assinados com uma chave inválida, causando rejeição por resolvers como o Google Public DNS.

- **Tipo de incidente:** Erro de configuração operacional (falha no gerenciamento de chave DNSSEC).  
- **Setor afetado:** Infraestrutura de internet (Registro de Domínios .br).  
- **Impacto:** Estratégico – interrupção de acesso a inúmeros sites .br.  
- **Entidade afetada:** NIC.br / Registro.br.  
- **Região:** Brasil.  
- **Ano:** 2020.  
- **Fontes:** [NIC.br](https://nic.br).  

---

## 2021: Ataque a Órgãos Federais via Falha de DNS (Caso ConecteSUS e Outros)

**Resumo:**  
Em dezembro de 2021, ataques cibernéticos atingiram órgãos do governo federal brasileiro, incluindo o Ministério da Saúde. A falha no DNS de uma fornecedora de internet foi explorada como vetor inicial.

- **Tipo de incidente:** Ataque cibernético (exploração de vulnerabilidade de DNS/network de provedor).  
- **Setor afetado:** Governo federal.  
- **Impacto:** Estratégico – violações de dados governamentais, indisponibilidade de serviços públicos.  
- **Entidades afetadas:** Ministério da Saúde (ConecteSUS), Ministério da Economia, entre outros.  
- **Região:** Brasil.  
- **Ano:** 2021.  
- **Fonte:** [IstoÉ Dinheiro](https://www.istoedinheiro.com.br).  
